# 07 One-Click LexAI Runner (Stable + UI-Link Safe)

This notebook is hardened for Databricks workspace-notebook paths and driver-proxy URL behavior.

Important:
- `http://127.0.0.1:<port>` works only inside this notebook runtime.
- For browser tabs, use the printed `driver-proxy` URLs from Cell 7 / Cell 9.


In [ ]:
# CELL 1: Runtime Flags
AUTO_INSTALL_MISSING = True
RUN_SMOKE_TEST = True
START_FASTAPI = True
START_STREAMLIT = False

# Set True only when import stack is broken and you accept manual rerun from Cell 1.
FORCE_REPAIR_IMPORT_STACK = False

FASTAPI_PORT = 8000
STREAMLIT_PORT = 8501

REPO_DIR_OVERRIDE = "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform"

SMOKE_TEST_QUERIES = [
    "Penalty for not wearing helmet in short within 120 words",
    "What does section 129 say in detail within 180 words",
]

print("[CELL 1] Flags loaded")
print({
    "AUTO_INSTALL_MISSING": AUTO_INSTALL_MISSING,
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "START_FASTAPI": START_FASTAPI,
    "START_STREAMLIT": START_STREAMLIT,
    "FORCE_REPAIR_IMPORT_STACK": FORCE_REPAIR_IMPORT_STACK,
    "FASTAPI_PORT": FASTAPI_PORT,
    "STREAMLIT_PORT": STREAMLIT_PORT,
    "REPO_DIR_OVERRIDE": REPO_DIR_OVERRIDE,
})


In [ ]:
# CELL 2: Resolve repo path safely
import os
import sys
from pathlib import Path
from datetime import datetime


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def _repo_has_required_files(repo_dir: Path) -> bool:
    return (
        (repo_dir / "apps" / "fastapi_app.py").exists()
        and (repo_dir / "apps" / "lexai06_notebook_adapter.py").exists()
        and (repo_dir / "notebooks" / "07_one_click_lexai_runner.ipynb").exists()
    )


def _safe_walk_for_repo(root: Path):
    skip_dirs = {"__pycache__", ".git", ".ipynb_checkpoints"}

    def _onerror(_err):
        return None

    for dirpath, dirnames, _ in os.walk(root, topdown=True, onerror=_onerror):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        p = Path(dirpath)
        try:
            if _repo_has_required_files(p):
                return p
        except Exception:
            continue
    return None


def _context_repo_guess():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        nb_path = ctx.notebookPath().get()  # /Users/<email>/<repo>/notebooks/07_...
        if not nb_path:
            return None
        pp = Path(nb_path)
        repo_ws = Path("/Workspace") / Path(*pp.parent.parent.parts[1:])
        if repo_ws.exists() and _repo_has_required_files(repo_ws):
            return repo_ws
    except Exception:
        pass
    return None


def resolve_repo_dir() -> Path:
    if REPO_DIR_OVERRIDE and str(REPO_DIR_OVERRIDE).strip():
        p = Path(REPO_DIR_OVERRIDE.strip())
        if p.exists() and _repo_has_required_files(p):
            return p

    cwd = Path(os.getcwd()).resolve()
    for cand in [cwd] + list(cwd.parents):
        try:
            if _repo_has_required_files(cand):
                return cand
        except Exception:
            continue

    g = _context_repo_guess()
    if g is not None:
        return g

    for root in [Path("/Workspace/Repos"), Path("/Workspace/Users"), Path("/Workspace")]:
        if not root.exists():
            continue
        hit = _safe_walk_for_repo(root)
        if hit is not None:
            return hit

    raise FileNotFoundError("Could not locate repo root. Set REPO_DIR_OVERRIDE to your repo path.")


REPO_DIR = resolve_repo_dir()
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

log(f"Repo root: {REPO_DIR}")
print("[CELL 2] OK")


In [ ]:
# CELL 3: Dependency preflight + import sanity (no forced restart)
import importlib
import importlib.metadata as ilm
import subprocess
import sys

REQ_FILE = Path("apps/requirements.txt")
if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

required_specs = [
    "fastapi",
    "uvicorn",
    "streamlit",
    "requests",
    "pydantic",
    "sentence-transformers",
    "transformers>=4.30.0",
    "accelerate>=0.20.0",
    "mlflow",
    "databricks-sdk",
    "typing_extensions>=4.6.0",
]


def _pkg_name(spec: str) -> str:
    for sep in [">=", "==", "<=", "~=", ">", "<"]:
        if sep in spec:
            return spec.split(sep)[0].strip()
    return spec.strip()


missing_specs = []
for spec in required_specs:
    try:
        ilm.version(_pkg_name(spec))
    except Exception:
        missing_specs.append(spec)

print("[CELL 3] Missing specs:", missing_specs)

if missing_specs and AUTO_INSTALL_MISSING:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)] + missing_specs
    print("[CELL 3] Installing missing specs...")
    subprocess.check_call(cmd)
elif missing_specs and not AUTO_INSTALL_MISSING:
    raise RuntimeError(f"Missing packages: {missing_specs}. Set AUTO_INSTALL_MISSING=True")

# Remove stale partially loaded modules before import tests.
for k in list(sys.modules.keys()):
    if k.startswith(("accelerate", "transformers")):
        del sys.modules[k]

try:
    import typing_extensions
    importlib.reload(typing_extensions)
    _ = typing_extensions.TypeIs

    import transformers
    import accelerate
    print("[CELL 3] Import sanity OK")
except Exception as e:
    print("[CELL 3] Import sanity failed:", e)
    if FORCE_REPAIR_IMPORT_STACK:
        repair_cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "typing_extensions>=4.6.0",
            "accelerate>=0.20.0",
            "transformers>=4.30.0",
        ]
        subprocess.check_call(repair_cmd)
        raise RuntimeError("Repair install complete. Run dbutils.library.restartPython(), then rerun from Cell 1.")
    raise

print("[CELL 3] OK")


In [ ]:
# CELL 4: Spark / context / browser-link base
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
print("[CELL 4] Spark session ready:", bool(spark))

cluster_id = "unknown"
org_id = "unknown"
workspace_url = "unknown"
compute_mode = "unknown"
DRIVER_PROXY_BASE = ""
DRIVER_PROXY_SUPPORTED = False


def _safe_conf(key: str, default: str = ""):
    try:
        v = spark.conf.get(key)
        if v and str(v).strip():
            return str(v).strip()
    except Exception:
        pass
    return default


def _opt_to_str(opt):
    try:
        if hasattr(opt, "isDefined") and opt.isDefined():
            return str(opt.get())
    except Exception:
        pass
    try:
        return str(opt.get())
    except Exception:
        pass
    return ""


def _ctx_tag(ctx, key: str):
    try:
        tags = ctx.tags()
        return _opt_to_str(tags.get(key))
    except Exception:
        pass
    try:
        tags = ctx.tags()
        return str(tags.apply(key))
    except Exception:
        pass
    return ""


ctx = None
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
except Exception:
    pass

cluster_id = _safe_conf("spark.databricks.clusterUsageTags.clusterId", "") or (_ctx_tag(ctx, "clusterId") if ctx else "")
org_id = _safe_conf("spark.databricks.clusterUsageTags.orgId", "") or (_ctx_tag(ctx, "orgId") if ctx else "")
compute_mode = _safe_conf("spark.databricks.clusterUsageTags.clusterSource", "") or (_ctx_tag(ctx, "clusterSource") if ctx else "") or "unknown"

ws_from_conf = _safe_conf("spark.databricks.workspaceUrl", "")
ws_from_ctx = _opt_to_str(ctx.browserHostName()) if ctx else ""
ws_api = _opt_to_str(ctx.apiUrl()) if ctx else ""
workspace_url = ws_from_conf or ws_from_ctx
if (not workspace_url) and ws_api:
    workspace_url = ws_api.replace("https://", "").split("/")[0]

if not workspace_url:
    workspace_url = "unknown"
if not cluster_id:
    cluster_id = "unknown"
if not org_id:
    org_id = "unknown"

if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
    DRIVER_PROXY_BASE = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}"
    DRIVER_PROXY_SUPPORTED = True

print("[CELL 4] cluster_id:", cluster_id)
print("[CELL 4] org_id:", org_id)
print("[CELL 4] workspace_url:", workspace_url)
print("[CELL 4] compute_mode:", compute_mode)
print("[CELL 4] DRIVER_PROXY_SUPPORTED:", DRIVER_PROXY_SUPPORTED)
print("[CELL 4] DRIVER_PROXY_BASE:", DRIVER_PROXY_BASE or "unavailable")


In [ ]:
# CELL 4.5: Optional repair flow (no auto-restart)
if FORCE_REPAIR_IMPORT_STACK:
    raise RuntimeError(
        "FORCE_REPAIR_IMPORT_STACK=True. Run: dbutils.library.restartPython(), then rerun from Cell 1."
    )
else:
    print("[CELL 4.5] FORCE_REPAIR_IMPORT_STACK=False -> skipped.")


In [ ]:
# CELL 5: Initialize notebook-06 engine through adapter
from pathlib import Path
import importlib
import os
import apps.lexai06_notebook_adapter as _adapter

importlib.reload(_adapter)
NotebookEngine = _adapter.NotebookEngine

workspace_candidates = [
    "/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine",
    "/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine",
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
    str(Path(REPO_DIR) / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"),
    str(Path(REPO_DIR) / "apps" / "notebook_06_snapshot.ipynb"),
]

print("[CELL 5] Notebook candidates:")
for c in workspace_candidates:
    try:
        print(" -", c, "exists=", Path(c).exists())
    except Exception:
        print(" -", c, "exists=ERROR")

status = None
last_err = None
for cand in workspace_candidates:
    try:
        os.environ["LEXAI06_NOTEBOOK_PATH"] = cand
        engine = NotebookEngine(notebook_path=Path(cand))
        status = engine.initialize()
        print(f"[CELL 5] Initialized using candidate: {cand}")
        break
    except Exception as e:
        last_err = e
        print(f"[CELL 5] Candidate failed: {cand} -> {e}")

if status is None:
    raise RuntimeError(f"Engine initialization failed for all candidates. Last error: {last_err}")

print("[CELL 5] Engine initialized")
for k, v in status.items():
    print(f"  - {k}: {v}")

if not status.get("ready"):
    raise RuntimeError(f"Engine failed to initialize: {status}")


In [ ]:
# CELL 6: Smoke test
if RUN_SMOKE_TEST:
    print("[CELL 6] Running smoke tests...")
    for idx, q in enumerate(SMOKE_TEST_QUERIES, start=1):
        print("=" * 90)
        print(f"[{idx}] Query: {q}")
        out = engine.answer_query(q)
        print("Mode:", out.get("mode"))
        print("Source:", out.get("source"))
        print("Confidence:", out.get("confidence"))
        print("Sections:", out.get("sections", []))
        print("Citations:", out.get("citations", [])[:5])
        print("Latency:", out.get("latency_ms", {}))
        print("Answer:")
        print(out.get("answer", ""))
    print("[CELL 6] Smoke tests done")
else:
    print("[CELL 6] RUN_SMOKE_TEST=False -> skipped")


In [ ]:
# CELL 7: Start FastAPI and print exact browser links
import threading
import time
import requests
import uvicorn

FASTAPI_SERVER = globals().get("FASTAPI_SERVER")
FASTAPI_THREAD = globals().get("FASTAPI_THREAD")
API_BROWSER_HEALTH_URL = ""
API_BROWSER_DOCS_URL = ""


def _wait_api_local(port: int, timeout_sec: int = 60):
    start = time.time()
    url = f"http://127.0.0.1:{port}/health"
    while (time.time() - start) < timeout_sec:
        try:
            r = requests.get(url, timeout=2)
            if r.status_code == 200:
                return True, r.json()
        except Exception:
            pass
        time.sleep(1)
    return False, {}


if START_FASTAPI:
    if FASTAPI_THREAD is None or not FASTAPI_THREAD.is_alive():
        from apps.fastapi_app import app
        config = uvicorn.Config(app, host="0.0.0.0", port=int(FASTAPI_PORT), log_level="info")
        FASTAPI_SERVER = uvicorn.Server(config)
        FASTAPI_THREAD = threading.Thread(target=FASTAPI_SERVER.run, daemon=True)
        FASTAPI_THREAD.start()
        globals()["FASTAPI_SERVER"] = FASTAPI_SERVER
        globals()["FASTAPI_THREAD"] = FASTAPI_THREAD
        print(f"[CELL 7] FastAPI start requested on 0.0.0.0:{FASTAPI_PORT}")
    else:
        print(f"[CELL 7] FastAPI already running on port {FASTAPI_PORT}")

    ok, body = _wait_api_local(int(FASTAPI_PORT), timeout_sec=60)
    print("[CELL 7] Local health check:", "OK" if ok else "FAILED")
    if body:
        print(body)

    print("[CELL 7] Local-only URL (inside notebook):", f"http://127.0.0.1:{FASTAPI_PORT}/health")
    print("[CELL 7] Do NOT open 127.0.0.1 in browser tab.")

    if DRIVER_PROXY_SUPPORTED:
        API_BROWSER_HEALTH_URL = f"{DRIVER_PROXY_BASE}/{FASTAPI_PORT}/health"
        API_BROWSER_DOCS_URL = f"{DRIVER_PROXY_BASE}/{FASTAPI_PORT}/docs"
        print("[CELL 7] Browser URL (health):", API_BROWSER_HEALTH_URL)
        print("[CELL 7] Browser URL (docs):", API_BROWSER_DOCS_URL)
        try:
            displayHTML(f'<a href="{API_BROWSER_DOCS_URL}" target="_blank">Open FastAPI Docs</a>')
        except Exception:
            pass
    else:
        print("[CELL 7] Driver proxy unsupported in this compute context. Use all-purpose cluster to open browser tabs.")
else:
    print("[CELL 7] START_FASTAPI=False -> skipped")


In [ ]:
# CELL 8: FastAPI smoke call
import requests

if START_FASTAPI:
    try:
        h = requests.get(f"http://127.0.0.1:{FASTAPI_PORT}/health", timeout=30)
        print("[CELL 8] local /health status:", h.status_code)
        print(h.json())

        payload = {
            "query": "What is the penalty for not wearing a helmet?",
            "style": "short",
            "word_limit": 120,
        }
        r = requests.post(f"http://127.0.0.1:{FASTAPI_PORT}/v1/legal/answer", json=payload, timeout=180)
        print("[CELL 8] local /v1/legal/answer status:", r.status_code)
        try:
            body = r.json()
            print("[CELL 8] answer preview:", body.get("answer", "")[:500])
        except Exception:
            print("[CELL 8] raw response:", r.text[:500])

        if API_BROWSER_DOCS_URL:
            print("[CELL 8] Open in browser:", API_BROWSER_DOCS_URL)
    except Exception as e:
        print("[CELL 8] API call failed:", e)
else:
    print("[CELL 8] START_FASTAPI=False -> skipped")


In [ ]:
# CELL 9: Optional Streamlit start (blocking) with browser URL
import subprocess

STREAMLIT_BROWSER_URL = ""

if START_STREAMLIT:
    os.environ["LEXAI_API_BASE_URL"] = f"http://127.0.0.1:{FASTAPI_PORT}"
    cmd = [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "apps/streamlit_app.py",
        "--server.port", str(STREAMLIT_PORT),
        "--server.address", "0.0.0.0",
    ]
    print("[CELL 9] Starting Streamlit:", " ".join(cmd))
    print("[CELL 9] Do NOT open localhost in browser tab.")

    if DRIVER_PROXY_SUPPORTED:
        STREAMLIT_BROWSER_URL = f"{DRIVER_PROXY_BASE}/{STREAMLIT_PORT}/"
        print("[CELL 9] Browser Streamlit URL:", STREAMLIT_BROWSER_URL)
        try:
            displayHTML(f'<a href="{STREAMLIT_BROWSER_URL}" target="_blank">Open Streamlit UI</a>')
        except Exception:
            pass
    else:
        print("[CELL 9] Driver proxy unsupported in this compute context. Use all-purpose cluster.")

    subprocess.call(cmd)
else:
    print("[CELL 9] START_STREAMLIT=False -> skipped")


In [ ]:
# CELL 10: Stop helper
if "FASTAPI_SERVER" in globals() and globals().get("FASTAPI_SERVER") is not None:
    globals()["FASTAPI_SERVER"].should_exit = True
    print("[CELL 10] FastAPI stop requested")
else:
    print("[CELL 10] FastAPI was not running")
